# Fase 2 — Limpieza y Feature Engineering

Imputación lógica de fechas, cálculo de `inventory_age` y margen de contribución.

OBJETIVO: MAXIMIZAR LIQUIDEZ DE LA EMPRESA ELIMINANDO EL STOCK MUERTO Y OPTIMIZANDO  LA INVERSION EN LOS PRODUCTOS QUE REALMENTE GENERAN MARGEN NETO TRAS DEVOLUCIONES

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/raw_master_data.parquet')
df.head()

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index
0,310417,1,Tops & Tees,Seven7,27.048,NaN,None,2025-04-25 06:18:30+00:00,2025-06-18 18:44:30+00:00,NaT,54,NaN,739.24829,0.74761,8,0,0.073047
1,219939,1,Tops & Tees,Seven7,27.048,49.0,Shipped,2020-10-30 14:05:00+00:00,NaT,NaT,2007,21.952,739.24829,0.74761,8,1,2.714920
2,219938,1,Tops & Tees,Seven7,27.048,NaN,None,2026-04-06 03:33:34+00:00,2026-04-23 15:45:34+00:00,NaT,17,NaN,739.24829,0.74761,8,0,0.022996
3,114521,1,Tops & Tees,Seven7,27.048,49.0,Complete,2024-06-19 03:55:00+00:00,NaT,NaT,679,21.952,739.24829,0.74761,8,1,0.918501
4,310419,1,Tops & Tees,Seven7,27.048,NaN,None,2020-06-10 16:31:00+00:00,NaT,NaT,2148,NaN,739.24829,0.74761,8,1,2.905654


In [67]:
df.dtypes # aqui verificamos que los tipos de datos coincidan, caso contrario se le dara el formato correcto

inventory_item_id                        int64
product_id                               int64
category                                object
brand                                   object
cost                                   float64
sale_price                             float64
status                                  object
inv_created_at             datetime64[ns, UTC]
sold_at                    datetime64[ns, UTC]
returned_at                datetime64[ns, UTC]
days_in_inventory                        int64
unit_margin                            float64
cat_avg_days_in_inv                    float64
cost_percentile_in_cat                 float64
total_sku_stock_history                  int64
is_dead_stock                            int64
inventory_age_index                    float64
dtype: object

In [68]:
# Verificamos integridad de datos, relacionado con fechas
# logica de fecha: cualquier fila donde el producto se venda antes de ser creado debe ser auditado
df[(df["inv_created_at"]>df["sold_at"]) | (df["returned_at"]<df["sold_at"])]

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index


No se muestran errores respecto a la integridad de fechas

In [69]:
# Verificamos filas donde si un producto es retornado, su status debe ser returned (cualqueir otro estado es erroneo)
df[df["returned_at"].notna() & (df["status"] != "Returned")]

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index


No se encontraron filas con errores logicos

# IMPUTACION
Desde este punto de la limpieza iniciamos con la imputacion de valores

## PRIMER OBJETIVO: `Sale_price` y `unit_margin`

Tanto sale_price y unit_margin son valores nulos para productos que nunca fueron vendidos, dado que nuestro objetivo es ver las lagunas en productos que no generan valor. Se inputaran sale_price y unit_margin con nuevas columnas `estimated_sale_price` y `estimated_unit_margin`

### Principales metodos a considerar apriori

- Inputacion mediante media de product_id
- Inputacion mediante mediana de product_id
- Casos extremos media o mediana por category

### Problemas
 - Los outliers generan sensibilidad en la media, por lo tanto sera necesario hacer un analisis de outliers e histograma (para ver la estabilidad de los precios de cada producto) y finalmente decidir el metodo estadistico correcto para su inputacion.

 #### Consideraciones
 Se considera hacer una funcion de limpieza en src/ que audite n_sales, median_price, mean_price, std_price, IQR y cv para automatizar el proceso de inputacion

In [70]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent)) # dado que el notebook esta en notebooks/ necesitamos resolver el path
from src.limpieza import impute_inventory_pricing #importamos la funcion de limpieza para este paso del proyecto

df_imputado = impute_inventory_pricing(df) #aplicamos sobre el df completo y verificamos la imputacion
df_imputado

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin
0,310417,1,Tops & Tees,Seven7,27.048,NaN,None,2025-04-25 06:18:30+00:00,2025-06-18 18:44:30+00:00,NaT,54,NaN,739.24829,0.747610,8,0,0.073047,31.950001,CATEGORY_MEDIAN,4.902001
1,219939,1,Tops & Tees,Seven7,27.048,49.0,Shipped,2020-10-30 14:05:00+00:00,NaT,NaT,2007,21.952,739.24829,0.747610,8,1,2.714920,49.000000,REAL,21.952000
2,219938,1,Tops & Tees,Seven7,27.048,NaN,None,2026-04-06 03:33:34+00:00,2026-04-23 15:45:34+00:00,NaT,17,NaN,739.24829,0.747610,8,0,0.022996,31.950001,CATEGORY_MEDIAN,4.902001
3,114521,1,Tops & Tees,Seven7,27.048,49.0,Complete,2024-06-19 03:55:00+00:00,NaT,NaT,679,21.952,739.24829,0.747610,8,1,0.918501,49.000000,REAL,21.952000
4,310419,1,Tops & Tees,Seven7,27.048,NaN,None,2020-06-10 16:31:00+00:00,NaT,NaT,2148,NaN,739.24829,0.747610,8,1,2.905654,31.950001,CATEGORY_MEDIAN,4.902001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487611,371401,29120,Accessories,Boconi,29.484,NaN,None,2022-11-04 01:25:00+00:00,NaT,NaT,1272,NaN,732.70233,0.815054,28,1,1.736039,78.000000,PRODUCT_MEAN,48.516000
487612,262837,29120,Accessories,Boconi,29.484,78.0,Processing,2020-06-22 14:27:00+00:00,NaT,NaT,2137,48.516,732.70233,0.815054,28,1,2.916601,78.000000,REAL,48.516000
487613,71065,29120,Accessories,Boconi,29.484,NaN,None,2023-04-15 07:28:00+00:00,2023-05-28 11:17:00+00:00,NaT,43,NaN,732.70233,0.815054,28,0,0.058687,78.000000,PRODUCT_MEAN,48.516000
487614,42989,29120,Accessories,Boconi,29.484,78.0,Shipped,2024-01-20 10:17:00+00:00,NaT,NaT,830,48.516,732.70233,0.815054,28,1,1.132793,78.000000,REAL,48.516000


In [71]:
df_imputado[df_imputado["estimated_sale_price"].isna()] #Dado que el resultado esta vacio, todas las filas fueron imputadas

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin


In [72]:
df = df_imputado

# BUSQUEDA DE OUTLIERS
Dado que ya se imputo de forma correcta, se muestra falta de valores nulos en columnas criticas y que todas las filas recibieron imputacion. Ahora sera necesario realizar el analisis de outliers.
La cantidad de datos es masiva, por ende productos tambien
## ESTRATEGIA
No se analizara cada producto, pero nuestro objetivo seran aquellos productos cuyo CV (coeficiente de variacion) es muy elevado o aquellos donde se priorizo la mediana sobre la media para la imputacion

In [73]:
df["imputation_source"].unique()
# no se muestran inputaciones por PRODUCT_MEDIAN

array(['CATEGORY_MEDIAN', 'REAL', 'PRODUCT_MEAN'], dtype=object)

# ANALISIS DE OUTLIERS EN days_in_inventory
Objetivo: Buscar productos que esten alejados de los IQR superior e inferior (no asumimos distribucion normal aun)
Finalmente deteccion de outliers temporales por categoria y de manera general

In [74]:
import importlib
from src import limpieza

importlib.reload(limpieza)

from src.limpieza import detect_inventory_age_outlier_categories

outliers_list = detect_inventory_age_outlier_categories(df)
outliers_list

[]

Resultados: Por un lado es positivo ninguna categoria tiene outliers respecto al tiempo en tienda, sin embargo quita la oportunidad de mostrar una imputacion mas profunda

In [75]:
importlib.reload(limpieza)

from src.limpieza import detect_inventory_cost_outlier_categories

outliers_list = detect_inventory_cost_outlier_categories(df)
outliers_list

['Accessories',
 'Active',
 'Blazers & Jackets',
 'Clothing Sets',
 'Dresses',
 'Fashion Hoodies & Sweatshirts',
 'Intimates',
 'Jeans',
 'Jumpsuits & Rompers',
 'Leggings',
 'Maternity',
 'Outerwear & Coats',
 'Pants',
 'Pants & Capris',
 'Plus',
 'Shorts',
 'Skirts',
 'Sleep & Lounge',
 'Socks',
 'Socks & Hosiery',
 'Suits',
 'Suits & Sport Coats',
 'Sweaters',
 'Swim',
 'Tops & Tees',
 'Underwear']

# Hallazgo
Cuando se usa el mismo script (modificado) para deteccion de outliers en `cost` notamos que si tenemos algunas categorias que presentan outliers respecto al costo de compra, ahora se procede a analizar

In [76]:
df_sospechosos = df[df["category"].isin(outliers_list)].copy()

# ahora obtenemos los outliers de cada categoria
outlier_rows = []

for category, group in df_sospechosos.groupby("category"):
    q1 = group["cost"].quantile(0.25)
    q3 = group["cost"].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    category_outliers = group[group["cost"] > upper_bound].copy()
    outlier_rows.append(category_outliers)

df_cost_outliers = pd.concat(outlier_rows, ignore_index=True)

df_cost_outliers

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin
0,266522,13597,Accessories,Ray-Ban,47.289439,121.879997,Processing,2021-05-01 04:42:00+00:00,NaT,NaT,1824,74.590559,732.70233,0.915412,12,1,2.489415,121.879997,REAL,74.590559
1,383820,13597,Accessories,Ray-Ban,47.289439,NaN,None,2023-12-02 18:05:37+00:00,2024-01-04 05:13:37+00:00,NaT,32,NaN,732.70233,0.915412,12,0,0.043674,121.879997,PRODUCT_MEAN,74.590559
2,60412,13597,Accessories,Ray-Ban,47.289439,121.879997,Processing,2020-07-21 05:15:00+00:00,NaT,NaT,2108,74.590559,732.70233,0.915412,12,1,2.877021,121.879997,REAL,74.590559
3,98936,13597,Accessories,Ray-Ban,47.289439,NaN,None,2025-07-26 15:44:47+00:00,2025-09-04 01:41:47+00:00,NaT,39,NaN,732.70233,0.915412,12,0,0.053228,121.879997,PRODUCT_MEAN,74.590559
4,98937,13597,Accessories,Ray-Ban,47.289439,NaN,None,2022-03-09 12:13:00+00:00,NaT,NaT,1512,NaN,732.70233,0.915412,12,1,2.063594,121.879997,PRODUCT_MEAN,74.590559
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27990,281375,26450,Underwear,Zimmerli of Switzerland,45.639000,NaN,None,2026-01-21 20:12:32+00:00,2026-02-14 06:10:32+00:00,NaT,23,NaN,738.72135,0.992641,14,0,0.031135,99.000000,PRODUCT_MEAN,53.361000
27991,171010,26450,Underwear,Zimmerli of Switzerland,45.639000,99.000000,Cancelled,2023-12-24 10:57:00+00:00,NaT,NaT,857,53.361000,738.72135,0.992641,14,1,1.160113,99.000000,REAL,53.361000
27992,171009,26450,Underwear,Zimmerli of Switzerland,45.639000,NaN,None,2024-06-03 06:59:53+00:00,2024-07-16 05:45:53+00:00,NaT,42,NaN,738.72135,0.992641,14,0,0.056855,99.000000,PRODUCT_MEAN,53.361000
27993,73345,26450,Underwear,Zimmerli of Switzerland,45.639000,99.000000,Complete,2025-10-08 04:36:00+00:00,NaT,NaT,203,53.361000,738.72135,0.992641,14,1,0.274799,99.000000,REAL,53.361000
